# Day 9 · Urban-rural — เมืองกับชนบทเห็นฟ้าเดียวกันไหม

**#30DayMapChallenge 2026 · วันที่ 9 พ.ย. 2569** · โจทย์: *"Urban or rural. Cities and countryside."*

สมมติฐานตั้งต้น "เมืองสว่าง ชนบทมืด" **ไม่จริง**กับข้อมูลชุดนี้:
Spearman(แรงกดดันเมือง, ระยะสู่ฟ้ามืด) = **+0.038** และ Spearman(ระยะที่ต้องเดินทาง, log ประชากรเมือง) = **+0.006**
แต่ hex ที่ทั้งใกล้ประชากร 500,000 คนและมืดพร้อมกัน = **0 จาก 1,738 บล็อก**
ปัญหาจึงไม่ใช่เมือง vs ชนบท แต่คือ *ที่มืดไม่ได้อยู่ใกล้คนที่อยากใช้*

> บน Colab: สคริปต์ดึง geojson จาก GitHub และโหลด shapefile Natural Earth เอง


### ตั้งค่า + shim (Colab/local) + helper haversine

In [ ]:
# -*- coding: utf-8 -*-
"""
Day 9 · Urban-rural — "Urban or rural. Cities and countryside."

สมมติฐานตั้งต้น (ของทุกคนรวมถึงผม): เมืองสว่าง ชนบทมืด → ความมืดคือเรื่องของชนบท
ข้อเท็จจริงจากข้อมูลชุดนี้: ไม่จริง — แรงกดดันเมืองกับระยะสู่ฟ้ามืดแทบไม่เกี่ยวกัน
(Spearman +0.04 ระดับ hex, +0.006 ระดับเมือง) แต่ "ที่ที่ทั้งมืดและใกล้คน" แทบไม่มีอยู่จริง
(hex ที่ใกล้ประชากร 500,000 คนใน 30 กม. และมืดพร้อมกัน = 0 จาก 1,738)

ปัญหาจึงไม่ใช่ "เมือง vs ชนบท" แต่คือ "ที่มืดไม่ได้อยู่ใกล้คนที่อยากใช้มัน"
"""
import os, json, math, time, zipfile, urllib.request
import numpy as np, pandas as pd
import geopandas as gpd
from shapely.geometry import shape, Point, mapping
from pyproj import CRS
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.collections import PolyCollection

T0 = time.time()
COLAB = not os.path.isdir(r"D:\GIS30Days\geo-notes")
WORK = os.getcwd() if COLAB else r"D:\GIS30Days"
REPO = os.path.join(WORK, "geo-notes")
CACHE = os.path.join(WORK, "day09_cache") if COLAB else os.path.join(WORK, "cache", "day06")
OUT = os.path.join(WORK, "out", "day09")
os.makedirs(CACHE, exist_ok=True); os.makedirs(OUT, exist_ok=True)

def ensure_shp(stem, sub):
    shp = os.path.join(CACHE, stem + ".shp")
    if not os.path.exists(shp):
        zp = os.path.join(CACHE, stem + ".zip")
        urllib.request.urlretrieve(f"https://naciscdn.org/naturalearth/10m/{sub}/{stem}.zip", zp)
        zipfile.ZipFile(zp).extractall(CACHE)
    return shp
def repo_file(rel):
    dst = os.path.join(CACHE, os.path.basename(rel))
    if COLAB and not os.path.exists(dst):
        urllib.request.urlretrieve("https://raw.githubusercontent.com/thammarat-ai/geo-notes/main/" + rel, dst)
    return dst if COLAB else os.path.join(REPO, rel)

for f in sorted(os.listdir(CACHE)):
    if f.endswith(".ttf"): font_manager.fontManager.addfont(os.path.join(CACHE, f))
plt.rcParams["font.family"] = "Trirong"
log = lambda m: print(f"[{time.time()-T0:5.1f}s] {m}", flush=True)

EQ = CRS.from_proj4("+proj=eqdc +lat_0=14 +lon_0=101 +lat_1=8 +lat_2=20 +units=m +no_defs +ellps=WGS84")
BG, INK = "#0d1522", "#e8e3d3"
COL = {"RD": "#2dd4bf", "RB": "#1e3a5f", "UD": "#f97316", "UB": "#f4566a"}

def hex_rings(g):
    for poly in (g.geoms if g.geom_type.startswith("Multi") else [g]):
        yield np.asarray(poly.exterior.coords)

def hav_km(a, b):
    la1, lo1 = np.radians(a[:, 1][:, None]), np.radians(a[:, 0][:, None])
    la2, lo2 = np.radians(b[:, 1][None, :]), np.radians(b[:, 0][None, :])
    d = np.sin((la2 - la1) / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(d))

### 1. ข้อมูล

In [ ]:
g3 = json.load(open(repo_file("posts/day03-polygons/day03_hex_polygons.geojson"), encoding="utf-8"))
hexes = [shape(f["geometry"]) for f in g3["features"]]
km = np.array([f["properties"]["km_nearest"] for f in g3["features"]], float)
prov = np.array([str(f["properties"]["province"]) for f in g3["features"]])
cen = np.array([[h.centroid.x, h.centroid.y] for h in hexes])
places = gpd.read_file(ensure_shp("ne_10m_populated_places", "cultural"))
th = places[(places.ADM0NAME == "Thailand") & (places.POP_MAX > 0)].copy()
city = np.column_stack([th.LONGITUDE.values, th.LATITUDE.values])
pop = th.POP_MAX.values.astype(float)
cname = np.array([str(x) for x in th.NAME.values])
countries = gpd.read_file(ensure_shp("ne_10m_admin_0_countries", "cultural"))
TH = gpd.GeoSeries([countries[countries.ISO_A3 == "THA"].geometry.iloc[0]], crs="EPSG:4326").to_crs(EQ)[0]
log(f"hex {len(hexes)} · เมือง {len(th)} · ประชากรเมืองรวม {pop.sum():,.0f}")

### 2. สองแกน + สี่กลุ่ม

In [ ]:
URB_R, DARK_CUT = 30.0, 25.0
URB_CUT = 100_000                 # "เป็นเมือง" = มีประชากรเมือง 100k+ ในรัศมี 30 กม.
D = hav_km(cen, city)
urban_press = (D <= URB_R) @ pop
is_dark = km <= DARK_CUT
is_urban = urban_press >= URB_CUT
quad = np.select([is_urban & is_dark, is_urban & ~is_dark, ~is_urban & is_dark],
                 ["UD", "UB", "RD"], default="RB")
cnt = {q: int((quad == q).sum()) for q in ["UD", "UB", "RD", "RB"]}
sp_hex = float(pd.Series(urban_press).corr(pd.Series(km), method="spearman"))
log(f"สี่กลุ่ม: {cnt} · Spearman(แรงกดดันเมือง, ระยะสู่ฟ้ามืด) = {sp_hex:+.3f}")

ctd = hav_km(city, cen[is_dark]).min(axis=1)      # จากกลางเมือง → hex ที่มืด
order = np.argsort(ctd)
bkk = int(np.argmax(pop))
sp_city = float(pd.Series(ctd).corr(pd.Series(np.log1p(pop)), method="spearman"))
log(f"ระยะสู่ hex มืด: ใกล้สุด {cname[order[0]]} {ctd[order[0]]:.1f} กม. · "
    f"ไกลสุด {cname[order[-1]]} {ctd[order[-1]]:.1f} กม. · มัธยฐาน {np.median(ctd):.1f} กม. · "
    f"กรุงเทพ {ctd[bkk]:.1f} กม.")
log(f"Spearman(ระยะสู่ฟ้ามืด, log ประชากรเมือง) = {sp_city:+.3f} → ขนาดเมืองไม่ทำนายระยะทาง")

### 3. verification (กติกาของซีรีส์)

In [ ]:
assert len(quad) == len(km) and set(np.unique(quad)) == {"UD", "UB", "RD", "RB"}
assert np.isfinite(ctd).all() and ctd.min() >= 0
# sanity bound: ในประเทศเดียว ระยะเมือง→hex ที่ไกลที่สุดต้องไม่เกิน ~1,000 กม.
# (รอบก่อนพิมพ์สูตร haversine ผิดได้ 10,000 กม. เงียบ ๆ โดยไม่มี error ใด ๆ)
assert ctd.max() < 1000, f"ระยะ {ctd.max():.0f} กม. เป็นไปไม่ได้ — check haversine"
assert D.max() < 2500 and D.min() >= 0
assert cnt["UD"] == int((is_urban & is_dark).sum())
assert int(((urban_press >= 500_000) & is_dark).sum()) == 0, "เกณฑ์ 500k ต้องไม่มี hex ที่มืด"
assert abs(sp_hex) < 0.15 and abs(sp_city) < 0.15, "ความสัมพันธ์ต้องอ่อนตามที่รายงาน"
log("ตรวจแล้ว: สี่กลุ่มครบไม่ซ้ำ · ไม่มี hex ที่ทั้งใกล้คน 500k และมืด · ความสัมพันธ์อ่อนจริงตามรายงาน")

### 4. แผนที่หลัก

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 12), facecolor=BG); ax.set_facecolor(BG)
ax.set_aspect("equal"); ax.axis("off")
land_rings = [np.asarray(p.exterior.coords) for poly in (TH.geoms if TH.geom_type.startswith("Multi") else [TH]) for p in [poly]]
ax.add_collection(PolyCollection(land_rings, facecolor="#111a2b", edgecolor="#4b5563", lw=0.8))
hexp = gpd.GeoSeries(hexes, crs="EPSG:4326").to_crs(EQ)
for q, alpha in [("RB", 0.55), ("UB", 0.75), ("RD", 0.9), ("UD", 0.95)]:
    m = quad == q
    if m.any():
        ax.add_collection(PolyCollection([r for g in hexp[m] for r in hex_rings(g)],
                                         facecolor=COL[q], edgecolor=COL[q], lw=0.1, alpha=alpha))
city_xy = gpd.GeoSeries([Point(*c) for c in city], crs="EPSG:4326").to_crs(EQ)
for (x, y), d in zip(zip(city_xy.x.values, city_xy.y.values), ctd):
    ax.plot(x, y, marker="o", ms=float(np.clip(2.0 + np.sqrt(d) / 9, 2.0, 11)),
            color="#f8fafc", mec="#0d1522", mew=0.4, zorder=6)
for i in (order[0], order[-1], bkk):
    ax.annotate(cname[i] + " · " + f"ไกลฟ้ามืด {ctd[i]:.0f} กม.",
                (city_xy.x.values[i], city_xy.y.values[i]), textcoords="offset points",
                xytext=(8, 6), fontsize=8.2, color="#fca5a5", zorder=7)
bx, by, bx1, by1 = TH.bounds
ax.set_xlim(bx - 70_000, bx1 + 70_000); ax.set_ylim(by - 120_000, by1 + 30_000)
ax.set_title("Day 9 · Urban-rural — เมืองกับชนบทเห็นฟ้าเดียวกันไหม" + chr(10) +
             f"เส้นแบ่งนี้ทำนายความมืดไม่ได้ (Spearman {sp_hex:+.2f}) แต่ที่ที่มืดและใกล้คนแทบไม่มีอยู่จริง",
             color=INK, fontsize=12.5, pad=26, fontweight="bold")
ax.text(0.5, 1.012, f"hex ที่ทั้งใกล้ประชากร 500,000 คน (ใน 30 กม.) และมืด = 0 จาก {len(hexes)} บล็อก",
        transform=ax.transAxes, ha="center", color="#f97316", fontsize=9.5)
LEG = [(0.000, "RD", f"ชนบท·มืด ({cnt['RD']})"), (0.175, "RB", f"ชนบท·ยังไม่มืด ({cnt['RB']})"),
       (0.400, "UD", f"เมือง·มืด ({cnt['UD']})"), (0.575, "UB", f"เมือง·ยังไม่มืด ({cnt['UB']})"),
       (0.790, None, "เมือง (ขนาด = ระยะสู่ฟ้ามืด)")]
for xf, q, lab in LEG:
    ax.plot([xf], [-0.032], marker="s" if q else "o", color=COL[q] if q else "#f8fafc",
            ms=9 if q else 6, clip_on=False, transform=ax.transAxes, zorder=9)
    ax.text(xf + 0.014, -0.032, lab, transform=ax.transAxes, fontsize=7.2, color="#cbd5e1", va="center")
fig.savefig(os.path.join(OUT, "day09_urban_rural_preview.png"), dpi=150, facecolor=BG, bbox_inches="tight")
log("เขียน day09_urban_rural_preview.png")

### 5. scatter + ranking

In [ ]:
fig2, (a1, a2) = plt.subplots(1, 2, figsize=(13.6, 5.8), facecolor=BG, gridspec_kw=dict(width_ratios=[1, 1.15]))
for a in (a1, a2): a.set_facecolor("#111a2b")
for q in ["RB", "UB", "RD", "UD"]:
    m = quad == q
    a1.scatter(np.log10(np.maximum(urban_press[m], 1)), km[m], s=9, color=COL[q], alpha=0.8,
               label=f"{q} ({int(m.sum())})")
a1.axhline(DARK_CUT, color="#94a3b8", ls="--", lw=1.0)
a1.axvline(math.log10(URB_CUT), color="#94a3b8", ls="--", lw=1.0)
a1.text(math.log10(URB_CUT) + 0.03, DARK_CUT + 3, f"เกณฑ์ {URB_CUT//1000}k คน / {DARK_CUT:.0f} กม.",
        color="#94a3b8", fontsize=8)
a1.set_xlabel("log10 ประชากรเมืองในรัศมี 30 กม.", color="#cbd5e1")
a1.set_ylabel("กม. ถึงจุดฟ้ามืดในทะเบียน", color="#cbd5e1")
a1.set_title(f"hex 1,738 บล็อก — สองแกนนี้แทบไม่เกี่ยวกัน (Spearman {sp_hex:+.3f})", color=INK, fontsize=11)
a1.legend(frameon=False, labelcolor="#cbd5e1", fontsize=8, loc="upper right")
for s in a1.spines.values(): s.set_color("#4b5563")
a1.tick_params(colors="#94a3b8"); a1.grid(alpha=0.12, color="#4b5563")

sel = np.concatenate([order[-12:][::-1], order[:12]])
a2.barh(range(len(sel)), ctd[sel], color=["#f4566a" if v > 60 else "#2dd4bf" for v in ctd[sel]])
a2.set_yticks(range(len(sel)), [cname[i] for i in sel], fontsize=8)
a2.invert_yaxis()
a2.set_xlabel("ระยะจากกลางเมือง → hex ที่มืดที่สุดที่หาได้ (กม.)", color="#cbd5e1")
a2.set_title(f"12 เมืองไกลฟ้ามืดที่สุด (บน) vs 12 ที่ใกล้ที่สุด (ล่าง) — "
             f"ขนาดเมืองไม่ทำนาย (Spearman {sp_city:+.3f})", color=INK, fontsize=10.5)
for i, v in zip(range(len(sel)), ctd[sel]):
    a2.text(v + 1.2, i, f"{v:.0f}", va="center", fontsize=7.4, color="#cbd5e1")
for s in a2.spines.values(): s.set_color("#4b5563")
a2.tick_params(colors="#94a3b8"); a2.grid(axis="x", alpha=0.12, color="#4b5563")
fig2.tight_layout(); fig2.savefig(os.path.join(OUT, "day09_compare.png"), dpi=150, facecolor=BG)
log("เขียน day09_compare.png")

### 6. สถิติ + ไฟล์ข้อมูล

In [ ]:
area_km2 = gpd.GeoSeries(hexes, crs="EPSG:4326").to_crs(EQ).area / 1e6
stat = {
    "urban_radius_km": URB_R, "urban_cut_pop": URB_CUT, "dark_cut_km": DARK_CUT,
    "hex_total": int(len(hexes)),
    "quadrants": {q: {"hex": cnt[q], "area_km2": int(area_km2[quad == q].sum())} for q in ["UD", "UB", "RD", "RB"]},
    "spearman_hex_urban_vs_dark": round(sp_hex, 3),
    "spearman_city_distance_vs_pop": round(sp_city, 3),
    "hex_urban500k_and_dark": int(((urban_press >= 500_000) & is_dark).sum()),
    "city_to_dark_km": {"min": {"city": cname[order[0]], "km": round(float(ctd[order[0]]), 1)},
                        "median": round(float(np.median(ctd)), 1),
                        "max": {"city": cname[order[-1]], "km": round(float(ctd[order[-1]]), 1)},
                        "bangkok": {"city": cname[bkk], "km": round(float(ctd[bkk]), 1), "pop": int(pop[bkk])},
                        "cities_under_10km": int((ctd < 10).sum()),
                        "cities_over_100km": int((ctd > 100).sum())},
    "note": "ความมืดวัดด้วย proxy = ระยะถึงจุดอนุรักษ์ท้องฟ้ามืดในทะเบียน NARIT (ไม่ใช่ค่าวัดแสงจริง)"}
json.dump(stat, open(os.path.join(OUT, "day09_stats.json"), "w"), ensure_ascii=False, indent=1)
gj = {"type": "FeatureCollection", "features": [
    {"type": "Feature", "properties": {"province": str(prov[i]), "km_nearest": float(km[i]),
                                       "urban_pop_30km": int(urban_press[i]), "quadrant": str(quad[i])},
     "geometry": mapping(h)}
    for i, h in enumerate(hexes)]}
json.dump(gj, open(os.path.join(OUT, "day09_hex_classes.geojson"), "w"), ensure_ascii=False)
log(f"เขียน stats + geojson · รวมเวลา {time.time()-T0:.1f} วิ")